## Model Evaluation
Evaluate query execution, BKT predictions and ranking with the existing project metrics.

In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/hlo_recommender").is_dir())
os.chdir(ROOT)

from hlo_recommender.cli import parser, run
from hlo_recommender.io import read_jsonl, write_json, write_jsonl

SESSION_FILE = ROOT / "artifacts/notebooks/session.json"
SESSION = json.loads(SESSION_FILE.read_text()) if SESSION_FILE.exists() else {}


In [ ]:
DATA = Path(SESSION.get("data_directory", ROOT / "data/development"))
OUTPUT = ROOT / "artifacts/notebooks" / f"evaluation-{uuid4().hex[:8]}"
MODEL = Path(SESSION.get("bkt_model", ROOT / "examples/results/bkt_model.json"))
RUN_MT5 = False
{"data": DATA.name, "bkt_model": MODEL.name, "mt5_enabled": RUN_MT5}


### Query Accuracy
Measure answer-set correctness and execution validity on held-out questions.

In [ ]:
query_metrics = run(parser().parse_args(["evaluate", "queries", "--generator", "template", "--data", str(DATA / "questions.test.jsonl"), "--output", str(OUTPUT / "queries")]))
query_metrics


### Knowledge Tracing
Predict each held-out response before updating mastery; report log loss, Brier score and AUC.

In [ ]:
bkt_metrics = run(parser().parse_args(["evaluate", "bkt", "--model", str(MODEL), "--data", str(DATA / "assessments.test.jsonl"), "--output", str(OUTPUT / "bkt")]))
{key: bkt_metrics[key] for key in ("log_loss", "brier", "auc")}


### Controlled Experiment
Compare static and adaptive ranking using matched retrieval and family-level confidence intervals.

In [ ]:
import tomllib

experiment = tomllib.loads((ROOT / "configs/experiment.toml").read_text())
experiment["runtime"] = str(ROOT / experiment["runtime"])
experiment["test"] = str(DATA / "questions.test.jsonl")
experiment["output"] = str(OUTPUT / "experiment")
CONFIG_PATH = ROOT / "artifacts/notebooks/configs" / f"{OUTPUT.name}.toml"
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text("\n".join(f"{key} = {json.dumps(value)}" for key, value in experiment.items()) + "\n")
comparison = run(parser().parse_args(["experiment", "--config", str(CONFIG_PATH)]))
{arm: result["metrics"] for arm, result in comparison["comparisons"].items()}


### Neural Comparison
Run both query generators only when a trained checkpoint exists; synthetic scores do not measure educational benefit.

In [ ]:
if RUN_MT5:
    checkpoint = SESSION.get("mt5_checkpoint")
    if not checkpoint:
        raise FileNotFoundError("Complete mT5 fine-tuning before neural evaluation.")
    neural_queries = run(parser().parse_args(["evaluate", "queries", "--generator", "mt5", "--checkpoint", checkpoint, "--data", str(DATA / "questions.test.jsonl"), "--output", str(OUTPUT / "mt5_queries")]))
    neural_comparison = run(parser().parse_args(["experiment", "--config", str(CONFIG_PATH), "--generator", "both", "--checkpoint", checkpoint, "--output", str(OUTPUT / "neural_experiment")]))
    print(json.dumps(neural_comparison["comparisons"], indent=2))
else:
    print("Neural evaluation was not executed.")


In [ ]:
SESSION["evaluation_directory"] = str(OUTPUT)
write_json(SESSION_FILE, SESSION)
sorted(path.relative_to(OUTPUT).as_posix() for path in OUTPUT.rglob("*.json"))
